# **PostgreSQL services validation**

Este notebook valida os serviços `services/pubmed_collection.py` e `services/crossref_enrichment.py` e os scripts `scripts/ingest_sample_postgres.py` e `scripts/enrich_crossref_sample_postgres.py`.

Diferente do notebook 04, as coletas consultam o PubMed e o Crossref de verdade, com limites pequenos. A persistência transacional já foi validada no notebook 04; aqui o foco é a orquestração. Os `run_id` criados ficam em `created_run_ids` e são apagados na limpeza.

Pré-requisito: o container do `compose.yaml` em execução (`docker compose up -d`).

## **Imports**

In [ ]:
import re
import subprocess
import sys

from sqlalchemy import func, select

from bibliometric_rag_assistant import db_postgres as db_pg
from bibliometric_rag_assistant.config import PROJECT_ROOT
from bibliometric_rag_assistant.contracts import PubMedSearchArgs
from bibliometric_rag_assistant.search_strategy import (
    PUBMED_ACTIVE_VARIANT,
    PUBMED_QUERY,
    PUBMED_VARIANTS,
)
from bibliometric_rag_assistant.services.crossref_enrichment import enrich_crossref_run
from bibliometric_rag_assistant.services.pubmed_collection import (
    collect_pubmed,
    resolve_preset,
)

In [ ]:
engine = db_pg.create_postgres_engine()
db_pg.create_tables(engine)

created_run_ids = []


def count_run_rows(run_id: str) -> dict[str, int]:
    with engine.connect() as connection:
        return {
            table.name: connection.execute(
                select(func.count())
                .select_from(table)
                .where(table.c.run_id == run_id)
            ).scalar_one()
            for table in db_pg.metadata.sorted_tables
        }

## **1. Identificação do preset**

Só a expressão completa, com o período, identifica uma variante. A mesma variante sem o filtro de data já é uma busca independente.

In [ ]:
thematic_query = PUBMED_VARIANTS[PUBMED_ACTIVE_VARIANT]["query"]

assert resolve_preset(PUBMED_QUERY) == (
    PUBMED_ACTIVE_VARIANT,
    PUBMED_VARIANTS[PUBMED_ACTIVE_VARIANT]["version"],
)
assert resolve_preset(thematic_query) == (None, None)

## **2. Coleta com o preset**

In [ ]:
preset_result = collect_pubmed(
    PubMedSearchArgs(query=PUBMED_QUERY, limit=5),
    engine=engine,
)
created_run_ids.append(preset_result.run_id)

preset_result.model_dump(mode="json")

In [ ]:
preset_run_id = preset_result.run_id

assert preset_result.status == "success"
assert preset_result.retrieved_count == len(preset_result.pmids) == 5
assert preset_result.parsed_count == preset_result.saved_count == 5

with engine.connect() as connection:
    preset_run = db_pg.get_ingestion_run(connection, preset_run_id)
    preset_citations = db_pg.load_citations(connection, preset_run_id)
    esearch = db_pg.load_raw_snapshot(connection, preset_run_id, "esearch")
    efetch = db_pg.load_raw_snapshot(connection, preset_run_id, "efetch")

assert preset_run["query_variant"] == PUBMED_ACTIVE_VARIANT
assert preset_run["total_results"] == preset_result.total_results
assert count_run_rows(preset_run_id)["raw_snapshots"] == 2

assert {citation["pmid"] for citation in preset_citations} == set(preset_result.pmids)
assert esearch["payload"]["esearchresult"]["idlist"] == preset_result.pmids
assert efetch["payload"].lstrip().startswith("<?xml")
assert esearch["file_path"] is None and efetch["file_path"] is None

## **3. Busca independente e busca vazia**

Uma query própria, com período explícito, grava variante e versão como `NULL`. Uma busca sem resultados é um sucesso: salva a execução e o snapshot do ESearch, sem EFetch, e preserva os avisos da fonte.

In [ ]:
independent_result = collect_pubmed(
    PubMedSearchArgs(
        query=(
            '"temporomandibular joint"[Title/Abstract] AND arthroscopy[Title/Abstract] '
            'AND ("2020/01/01"[Date - Publication] : "2020/12/31"[Date - Publication])'
        ),
        limit=3,
    ),
    engine=engine,
)
created_run_ids.append(independent_result.run_id)

with engine.connect() as connection:
    independent_run = db_pg.get_ingestion_run(connection, independent_result.run_id)

assert independent_result.status == "success"
assert independent_run["query_variant"] is None
assert independent_run["query_version"] is None
assert count_run_rows(independent_result.run_id)["citations"] == independent_result.saved_count

In [ ]:
empty_result = collect_pubmed(
    PubMedSearchArgs(
        query=(
            '"temporomandibular joint"[Title/Abstract] '
            'AND ("1800/01/01"[Date - Publication] : "1800/12/31"[Date - Publication])'
        ),
        limit=5,
    ),
    engine=engine,
)
created_run_ids.append(empty_result.run_id)

assert empty_result.status == "success"
assert empty_result.total_results == empty_result.saved_count == 0
assert "outputmessages: No items found." in empty_result.warnings
assert count_run_rows(empty_result.run_id) == {
    "ingestion_runs": 1,
    "raw_snapshots": 1,
    "citations": 0,
    "citation_enrichments": 0,
}

empty_result.warnings

## **4. Enriquecimento e retomada**

In [ ]:
first_enrichment = enrich_crossref_run(preset_run_id, engine=engine)

first_enrichment["results"], first_enrichment["skipped"]

In [ ]:
results = first_enrichment["results"]

assert first_enrichment["skipped"] == 0
assert sum(results.values()) == first_enrichment["saved"] == 5

with engine.connect() as connection:
    enriched_pmids = db_pg.get_enriched_pmids(connection, preset_run_id, "crossref")
    gaps = db_pg.load_enrichment_gaps(connection, preset_run_id, "crossref")

# Os registros pertencem ao mesmo run_id e cobrem todas as citações
assert enriched_pmids == set(preset_result.pmids)
assert count_run_rows(preset_run_id)["citation_enrichments"] == 5
assert len(gaps) == results["missing_doi"] + results["not_found"]

Uma segunda chamada não consulta o Crossref de novo: os três status contam como processados.

In [ ]:
second_enrichment = enrich_crossref_run(preset_run_id, engine=engine)

assert second_enrichment["skipped"] == 5
assert second_enrichment["pending"] == second_enrichment["saved"] == 0
assert count_run_rows(preset_run_id)["citation_enrichments"] == 5

Uma execução inexistente é um erro; uma execução válida sem citações termina com contagens zeradas.

In [ ]:
try:
    enrich_crossref_run("validation_missing_run", engine=engine)
except LookupError:
    pass
else:
    raise AssertionError("Uma execução inexistente deveria gerar LookupError")

empty_enrichment = enrich_crossref_run(empty_result.run_id, engine=engine)

assert empty_enrichment["total_citations"] == empty_enrichment["saved"] == 0

## **5. Scripts pelo terminal**

Os scripts rodam como processos separados, com o mesmo Python deste kernel. O `run_id` impresso pela ingestão é usado no enriquecimento.

In [ ]:
def run_script(*args: str) -> str:
    completed = subprocess.run(
        [sys.executable, *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    assert completed.returncode == 0, completed.stderr

    return completed.stdout


ingest_output = run_script(
    "scripts/ingest_sample_postgres.py",
    "--query", '"temporomandibular joint"[Title/Abstract] AND ankylosis[Title/Abstract]',
    "--limit", "3",
)
script_run_id = re.search(r"^Run:\s+(\S+)", ingest_output, re.MULTILINE).group(1)
created_run_ids.append(script_run_id)

print(ingest_output)

In [ ]:
enrich_output = run_script(
    "scripts/enrich_crossref_sample_postgres.py",
    "--run-id", script_run_id,
)

assert "Preset:    none (independent search)" in ingest_output
assert count_run_rows(script_run_id) == {
    "ingestion_runs": 1,
    "raw_snapshots": 2,
    "citations": 3,
    "citation_enrichments": 3,
}

print(enrich_output)

## **6. Limpeza**

In [ ]:
with engine.begin() as connection:
    for table in reversed(db_pg.metadata.sorted_tables):
        connection.execute(
            table.delete().where(table.c.run_id.in_(created_run_ids))
        )

for created_run_id in created_run_ids:
    assert sum(count_run_rows(created_run_id).values()) == 0

engine.dispose()

created_run_ids